# Do AntiFold's per-residue tables line up with 1N8Z?

Two tables were written for the ADAPT calibration, one per condition, and every
`antifold` score in notebooks 03 and 03_antigen is a subtraction of two cells
out of them. A table shifted by one position still yields real-looking
log-odds - for the wrong residue - so this notebook runs
`antifold_score.check_wt` over both and prints what it found. It loads no model
and needs no GPU: the tables already exist, and this reads them.

It is also the standing answer to "is the +0.045 an index offset?". It is not
the answer to "is AntiFold any good" - that is notebook 03's job.

## Where each input comes from

Nothing this notebook reads is in git, which is why a fresh clone cannot run it:

| Input | Written by | Why it is not in git |
|---|---|---|
| `data/structures/1N8Z.pdb` | `pdb_download`, i.e. `get_structure` (`design_pipeline/api.py`) - one HTTP GET from `https://files.rcsb.org/download` (`design_pipeline/adapters/_pdb.py`). Or dropped in by hand; 703,809 bytes either way | `.gitignore` line 2, `data/*` |
| `runs/adapt_calibration/antifold/1N8Z_BA.csv` | upstream `antifold/main.py`, started as a subprocess by `antifold_score.run_antifold` with `--heavy_chain B --light_chain A --device cpu` and CUDA hidden. The name is computed, not read: `f"{pdb.stem}_{heavy}{light}.csv"` | `.gitignore` line 20, `runs/` |
| `runs/adapt_calibration_antigen/antifold/1N8Z_BAC.csv` | the same AntiFold binary, started by `run_antifold_antigen` - a helper local to cell 10 of `03_adapt_calibration_antigen.ipynb`, not a module - with `--antigen_chain C` added. With an antigen AntiFold switches to custom chain mode and renames the table, so that helper globs `<stem>_<heavy><light>*.csv` instead of predicting the name | the same `runs/` |
| `.../antifold/imgt_numbered/1N8Z.pdb` | AntiFold, beside each table. The labels in the tables are this copy's, not the input file's - 1N8Z's heavy chain has no position 10 - so it is what `pdb_posins` is mapped through | the same `runs/` |

Both notebooks pointed `out_dir` at a relative `runs/...`, resolved against the
repo root they ran in. So on the node both tables sit **inside the checkout**
and are untracked because `runs/` is ignored - that is why `git pull` brings the
notebooks and never the CSVs. Run this notebook where they were written.

1N8Z is the trastuzumab Fab bound to HER2 domain IV: chain B heavy (220
residues), chain A light (214), chain C HER2 (581). The antigen table therefore
has 1,015 rows to the Fab-only table's 434; chain C is read by neither scorer
and is skipped below.


In [ ]:
# --- parameters -------------------------------------------------------------
# The two tables, by the condition each one is. None of these paths is created
# here; the table above says what writes each.
TABLES = {
    "Fab only (notebook 03)": "runs/adapt_calibration/antifold/1N8Z_BA.csv",
    "HER2 chain C in the input (notebook 03_antigen)":
        "runs/adapt_calibration_antigen/antifold/1N8Z_BAC.csv",
}

# The local 1N8Z. None searches the places a copy usually lands. Nothing here
# downloads: a structure that arrived by another route is a different structure,
# and every position in both tables is an index into this one file.
PDB_PATH = None
HEAVY_CHAIN = "B"
LIGHT_CHAIN = "A"


In [ ]:
import os
import sys
from pathlib import Path

# Run from the repo root whatever the working directory, so the relative paths
# above mean one thing and `antifold_score` is importable.
root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
if not (root / "pyproject.toml").is_file():
    raise RuntimeError(f"no repo root above {Path.cwd()}: pyproject.toml not found")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from antifold_score import check_wt
from design_pipeline.matrices import IndexMismatchError

PDB_CANDIDATES = (
    Path("data/structures/1N8Z.pdb"),
    Path("data/1N8Z.pdb"),
    Path.home() / ".cache" / "design_pipeline" / "pdb" / "1n8z.pdb",
)
if PDB_PATH is not None:
    pdb = Path(PDB_PATH).expanduser()
    if not pdb.is_file():
        raise FileNotFoundError(f"PDB_PATH does not exist: {pdb}")
else:
    pdb = next((c for c in PDB_CANDIDATES if c.is_file()), None)
    if pdb is None:
        raise FileNotFoundError(
            "no 1N8Z found. Looked in:\n"
            + "\n".join(f"  {c}" for c in PDB_CANDIDATES)
            + "\nFetch it with design_pipeline.get_structure('1N8Z') or set PDB_PATH."
        )

print(f"repo root: {root}")
print(f"structure: {pdb}  ({pdb.stat().st_size:,} bytes)")
print()
print(f"{'condition':<50}{'table':>10}{'rows':>8}")
print("-" * 68)
present = {}
for condition, relative in TABLES.items():
    path = Path(relative)
    if path.is_file():
        rows = sum(1 for _ in path.open(encoding="utf-8")) - 1
        present[condition] = path
        print(f"{condition:<50}{'found':>10}{rows:>8,}")
    else:
        print(f"{condition:<50}{'ABSENT':>10}{'-':>8}")
if not present:
    print()
    print("Neither table is here. They are written where AntiFold ran - see the table")
    print("above - and `runs/` is gitignored, so they do not travel with the repo.")


In [ ]:
# --- the check, per table ----------------------------------------------------
# What it asserts, per antibody chain: AntiFold's own wild-type column, in file
# order, is the chain's sequence (`LogProbMatrix.assert_matches`); no other
# offset in -3..+3 fits as well; every `pdb_posins` maps to a position holding
# the residue the row claims; and the argmax of the 20 columns is the row's
# `top_res`, so the columns are the residues this package thinks they are.
#
# The equivalent command line, which is what to run when this notebook is not
# the convenient way:
#
#     python -m antifold_score.check_wt --table <the csv> \
#         --pdb data/structures/1N8Z.pdb --heavy B --light A
#
# `top_res` differing from `pdb_res` is not a failure - that is the model
# preferring another residue to the crystal's, which is a result. It is counted.
verdicts = {}
for condition, relative in TABLES.items():
    print("=" * 78)
    print(condition)
    print("=" * 78)
    path = Path(relative)
    if not path.is_file():
        print(f"not here: {path}")
        print("run this where it was written:")
        print(f"  python -m antifold_score.check_wt --table {relative} \\")
        print(f"      --pdb {pdb.as_posix()} --heavy {HEAVY_CHAIN} --light {LIGHT_CHAIN}")
        verdicts[condition] = None
        print()
        continue
    try:
        print("\n".join(check_wt.check(path, pdb, [HEAVY_CHAIN, LIGHT_CHAIN])))
        print("PASSED: the matrix describes the positions it claims to.")
        verdicts[condition] = True
    except (check_wt.CheckFailed, IndexMismatchError) as exc:
        print(f"MISALIGNED: {exc}")
        verdicts[condition] = False
    print()

print("=" * 78)
for condition, verdict in verdicts.items():
    state = {True: "passed", False: "MISALIGNED", None: "not run - table absent"}[verdict]
    print(f"{condition:<52}{state}")
checked = [v for v in verdicts.values() if v is not None]
print()
if not checked:
    print("Nothing was checked: neither table is on this machine, so this notebook says")
    print("nothing either way about an index offset.")
elif not all(checked):
    print("At least one table is misaligned. Every score taken off it is a real number")
    print("for the wrong residue, so the affected notebook's scores have to be re-run.")
elif len(checked) == len(verdicts):
    print("Both tables are aligned, so an index offset is not what either run's scores")
    print("are. Whatever the matrix lookup's +0.045 against the ADAPT ddG panel is, it is")
    print("the model's opinion of those four aspartates and not a misread row.")
else:
    absent = [c for c, v in verdicts.items() if v is None]
    print(f"The {len(checked)} table(s) read here are aligned. Not checked, so still open:")
    for condition in absent:
        print(f"  {condition}")


## If a table is missing

Neither can be regenerated without AntiFold itself - its own environment (Env B;
`pip install "biotite>=2.0" antifold` is `ResolutionImpossible`, see `INSTALL.md`)
and its 567 MB `models/model.pt`. CPU is fine: one structure is seconds, and the
whole antigen run was 63 s.

The Fab-only table, from this repo:

```python
from antifold_score import run_antifold
run_antifold(pdb, "B", "A", Path("runs/adapt_calibration/antifold"),
             Path(os.environ["ANTIFOLD_SRC"]))
```

The antigen table is not reachable from a module: `--antigen_chain` lives only
in `run_antifold_antigen` in cell 10 of `03_adapt_calibration_antigen.ipynb`, so
re-running that notebook's Part 2 is the way to get `1N8Z_BAC.csv` back. If the
antigen route earns a place in `antifold_score`, that helper is what moves.

A cheaper fix for the long run: both tables together are about 270 KB, and
`tests/outputs/` already keeps AntiFold results for exactly this purpose. Copied
in there, this check runs in CI and survives the node's scratch being wiped.
